# Deep RL Bootcamp — Mathematical Foundations & Experiments

> **A complete, executable top-to-bottom notebook** distilled from the Berkeley Deep RL Bootcamp lectures, OpenAI Spinning Up, and hands-on Gymnasium / Stable-Baselines3 practice.

**Design goals (no hand-waving):**
- Every entity is defined mathematically: the probability space, the action space, the state space, transition distributions, value functions.
- Every experiment is runnable end-to-end.
- Animations capture the *internal state* of the selected agent as it learns.
- Package choices and exact versions are justified in-notebook.

**Lecture lineage (Berkeley Deep RL Bootcamp 2017):**
1. Intro to MDPs and Exact Solution Methods — *Pieter Abbeel*
2. Sample-based Approximations and Fitted Learning — *Rocky Duan*
3. DQN + Variants — *Vlad Mnih*
4. Policy Gradients and Actor-Critic — *Pieter Abbeel*
5. Natural Policy Gradients, TRPO, PPO — *John Schulman*

---

## Notebook Map
| Part | Topic | Key Math |
|------|-------|----------|
| 0 | Setup & package justification | — |
| I | Probability & MDP formalism | $(\Omega,\mathcal F,\mathbb P)$, $\mathcal M=(\mathcal S,\mathcal A,\mathcal P,\mathcal R,\gamma)$ |
| II | Bandits & exploration | $\epsilon$-greedy, UCB1, Thompson |
| III | Exact solution methods | Value / Policy Iteration |
| IV | Model-free learning | MC, TD(0), SARSA, Q-Learning |
| V | Deep RL | DQN, REINFORCE, A2C, PPO |
| VI | Animations | Agent internal-state capture |

---
# Part 0 — Setup & Package Justification

Reinforcement learning code is notoriously version-sensitive: the Gym → Gymnasium API change (5-tuple `step`) and the Stable-Baselines3 v2 migration both break older notebooks. To guarantee this notebook runs **top-to-bottom**, we pin exact versions and justify each one.

| Package | Pinned Version | Why this package / version |
|---------|----------------|----------------------------|
| `numpy` | 1.26.4 | Array math, sampling from distributions. 1.26.x is the last NumPy 1.x line — avoids NumPy 2.0 ABI breaks that several RL wheels were not yet built against. |
| `scipy` | 1.11.4 | `scipy.stats.beta`/`norm` for the Thompson-sampling posteriors and policy-distribution plots. |
| `matplotlib` | 3.8.2 | Plotting **and** `FuncAnimation` for the agent-state animations. |
| `gymnasium` | 0.29.1 | The maintained successor to OpenAI Gym. Provides the canonical `(obs, reward, terminated, truncated, info)` API and the `FrozenLake`, `CartPole` environments we use. |
| `torch` | 2.1.2 | Neural network policies/value functions (REINFORCE from scratch + SB3 backend). CPU build is sufficient for MLP policies. |
| `stable-baselines3` | 2.3.2 | Battle-tested reference implementations of DQN / A2C / PPO. v2.x requires Gymnasium (not Gym), which is why we pin Gymnasium ≥ 0.28. |
| `pandas` | 2.1.4 | Rolling-window smoothing of learning curves. |

> **Reproducibility note:** every stochastic component is seeded with `SEED = 42`. Run the cells in order.

In [ ]:
# ── Exact, pinned installation. Run once, then restart the kernel if needed. ──
# These versions are mutually compatible (SB3 2.3.2 ⇒ gymnasium, torch ≥ 1.13).
!pip install -q \
    numpy==1.26.4 \
    scipy==1.11.4 \
    matplotlib==3.8.2 \
    pandas==2.1.4 \
    gymnasium==0.29.1 \
    torch==2.1.2 \
    "stable-baselines3==2.3.2"

print("Installation cell finished. If imports below fail, restart the kernel and re-run.")

In [ ]:
# ── Imports & global configuration ──
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle, Rectangle, FancyArrowPatch
from scipy.stats import beta as beta_dist, norm
from IPython.display import HTML, display

import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium.spaces import Box, Discrete

# Reproducibility: one seed to rule them all.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Animations rendered as inline HTML5 video.
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["figure.dpi"] = 110

print("Versions in use")
print(f"  numpy       {np.__version__}")
print(f"  torch       {torch.__version__}")
print(f"  gymnasium   {gym.__version__}")
print("Setup complete — seed =", SEED)

---
# Part I — Probability & MDP Formalism

Reinforcement learning is decision-making under uncertainty. Everything below is built on a single **probability space**, so we define it precisely before touching any algorithm.

## I.1 The probability space $(\Omega, \mathcal F, \mathbb P)$

- **Sample space $\Omega$** — the set of all possible *trajectories*
$$\tau = (s_0, a_0, r_1, s_1, a_1, r_2, \dots) \in \Omega.$$
A single outcome $\omega \in \Omega$ is one complete trajectory.

- **$\sigma$-algebra $\mathcal F$** — the measurable events, e.g. "the agent reaches the goal before $t=100$". Formally the smallest $\sigma$-algebra making every $S_t, A_t, R_t$ measurable.

- **Probability measure $\mathbb P$** — induced jointly by the start distribution $\rho_0$, the policy $\pi$, and the transition kernel $\mathcal P$:
$$\mathbb P_\pi(\tau) = \rho_0(s_0)\prod_{t\ge 0}\pi(a_t\mid s_t)\,\mathcal P(s_{t+1}\mid s_t,a_t).$$

Random variables on this space:
$$S_t:\Omega\to\mathcal S,\quad A_t:\Omega\to\mathcal A,\quad R_t:\Omega\to\mathbb R.$$

## I.2 The Markov Decision Process

$$\mathcal M = (\mathcal S,\ \mathcal A,\ \mathcal P,\ \mathcal R,\ \gamma)$$

| Symbol | Meaning | Formal type |
|--------|---------|-------------|
| $\mathcal S$ | state space | measurable set |
| $\mathcal A$ | action space | measurable set |
| $\mathcal P(s'\mid s,a)$ | transition kernel | conditional pmf/pdf, $\sum_{s'}\mathcal P=1$ |
| $\mathcal R(s,a)=\mathbb E[R_{t+1}\mid s,a]$ | reward function | $\mathcal S\times\mathcal A\to\mathbb R$ |
| $\gamma\in[0,1)$ | discount | scalar |

**Markov property** — the present state is a *sufficient statistic* for the future:
$$\mathbb P(S_{t+1}\mid S_t,A_t,S_{t-1},\dots,S_0)=\mathbb P(S_{t+1}\mid S_t,A_t).$$

## I.3 Action space $\mathcal A$ and the policy distribution

The **policy** is the conditional distribution the agent controls:

$$\pi_\theta(a\mid s)=\begin{cases}\operatorname{Categorical}\big(\operatorname{softmax} f_\theta(s)\big) & \mathcal A=\{0,\dots,n-1\}\ \text{(discrete)}\\[4pt]\mathcal N\!\big(\mu_\theta(s),\,\sigma_\theta(s)^2\big) & \mathcal A\subseteq\mathbb R^d\ \text{(continuous)}\end{cases}$$

The cell below renders these two distribution families explicitly.

In [ ]:
# ── Visualise the two policy distribution families + the Markov transition kernel ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) Discrete policy: a categorical distribution over 4 actions
ax = axes[0]
actions = ["←", "↓", "→", "↑"]
logits = np.array([0.4, 1.8, 2.4, 0.9])          # raw network outputs f_theta(s)
probs = np.exp(logits) / np.exp(logits).sum()    # softmax
ax.bar(actions, probs, color="#4C72B0", edgecolor="black")
for i, p in enumerate(probs):
    ax.text(i, p + 0.01, f"{p:.2f}", ha="center", fontweight="bold")
ax.set_title("Discrete policy\n$\\pi(a|s)=\\mathrm{softmax}(f_\\theta(s))$")
ax.set_ylabel("probability")
ax.set_ylim(0, max(probs) + 0.1)

# (b) Continuous policy: Gaussian densities for different state-dependent means
ax = axes[1]
xs = np.linspace(-3, 3, 400)
for mu, sd, c in [(-1.0, 0.5, "#C44E52"), (0.0, 0.8, "#55A868"), (1.2, 0.4, "#8172B3")]:
    ax.plot(xs, norm.pdf(xs, mu, sd), color=c, lw=2, label=f"$\\mu$={mu}, $\\sigma$={sd}")
ax.set_title("Continuous policy\n$\\pi(a|s)=\\mathcal N(\\mu_\\theta(s),\\sigma_\\theta(s)^2)$")
ax.set_xlabel("action $a$"); ax.set_ylabel("density"); ax.legend(fontsize=8)

# (c) A valid transition kernel: each row is a probability distribution
ax = axes[2]
P = np.array([
    [0.0, 0.7, 0.2, 0.1],
    [0.6, 0.0, 0.3, 0.1],
    [0.1, 0.2, 0.0, 0.7],
    [0.1, 0.1, 0.8, 0.0],
])
im = ax.imshow(P, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels([f"s'{i}" for i in range(4)]); ax.set_yticklabels([f"s{i}" for i in range(4)])
ax.set_title("Transition kernel $\\mathcal P(s'|s,a)$\n(rows sum to 1)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{P[i,j]:.1f}", ha="center", va="center",
                color="white" if P[i, j] > 0.5 else "black", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout(); plt.show()

# Numerically verify the kernel is a proper conditional distribution
assert np.allclose(P.sum(axis=1), 1.0), "Each row of P must sum to 1"
print("Row sums of P:", P.sum(axis=1), "→ valid stochastic matrix")

## I.4 Return, value functions, and the Bellman equations

The **return** is the discounted sum of future rewards — a random variable on $\Omega$:
$$G_t = \sum_{k=0}^{\infty}\gamma^k R_{t+k+1}.$$
Discounting guarantees convergence when rewards are bounded ($|R|\le R_{\max}$): $|G_t|\le R_{\max}/(1-\gamma)$.

**State-value** and **action-value** functions are conditional expectations of the return:
$$V^\pi(s)=\mathbb E_\pi[G_t\mid S_t=s],\qquad Q^\pi(s,a)=\mathbb E_\pi[G_t\mid S_t=s,A_t=a].$$

Because of the Markov property, $G_t = R_{t+1} + \gamma G_{t+1}$, which gives the **Bellman expectation equation**:
$$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}\mathcal P(s'\mid s,a)\big[\mathcal R(s,a)+\gamma V^\pi(s')\big].$$

Taking the greedy maximum yields the **Bellman optimality equations**, whose unique fixed points are $V^\*$ and $Q^\*$:
$$V^\*(s)=\max_a\sum_{s'}\mathcal P(s'\mid s,a)\big[\mathcal R(s,a)+\gamma V^\*(s')\big],$$
$$Q^\*(s,a)=\sum_{s'}\mathcal P(s'\mid s,a)\big[\mathcal R(s,a)+\gamma\max_{a'}Q^\*(s',a')\big].$$

The cell below (1) shows how $\gamma$ reweights the future and (2) computes returns for a concrete reward stream by the backward recursion $G_t = R_{t+1}+\gamma G_{t+1}$.

In [ ]:
# ── (1) Discount weighting and (2) the backward return recursion ──
def compute_returns(rewards, gamma):
    # G_t = R_{t+1} + gamma * G_{t+1}, computed backward in O(T).
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    return out[::-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# (1) How much does a reward k steps away count?
k = np.arange(0, 50)
for g in [0.5, 0.9, 0.99, 1.0]:
    axes[0].plot(k, g ** k, lw=2, label=f"$\\gamma$={g}")
axes[0].set_title("Discount weight $\\gamma^k$ on a reward $k$ steps ahead")
axes[0].set_xlabel("steps into the future $k$"); axes[0].set_ylabel("$\\gamma^k$")
axes[0].legend(); axes[0].grid(alpha=0.3)

# (2) Concrete return computation
rewards = [1, 1, 1, 1, 0, 0, 1, 1, 0]
gamma = 0.9
G = compute_returns(rewards, gamma)
t = np.arange(len(rewards))
axes[1].bar(t, rewards, alpha=0.55, color="#C44E52", label="reward $R_{t+1}$")
axes[1].plot(t, G, "o-", color="#4C72B0", lw=2, label="return $G_t$")
axes[1].set_title(f"Return via $G_t=R_{{t+1}}+\\gamma G_{{t+1}}$  ($\\gamma$={gamma})")
axes[1].set_xlabel("time step $t$"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Sanity check against the closed-form geometric bound |G| <= Rmax/(1-gamma)
print(f"max |G_t| observed = {max(map(abs, G)):.3f}")
print(f"theoretical bound  = Rmax/(1-gamma) = {1/(1-gamma):.3f}")

---
# Part II — Bandits & the Exploration/Exploitation Trade-off

The **$k$-armed bandit** is the simplest non-trivial decision problem: a *single-state* MDP with $k$ actions. There is no transition dynamics to worry about, so it isolates the core difficulty: **we must estimate action values from noisy samples while deciding how much to explore.**

**Setup.** Each arm $a\in\{0,\dots,k-1\}$ has an unknown reward distribution with mean $q^\*(a)$. Pulling arm $a$ returns $R\sim\mathcal D_a$ with $\mathbb E[R]=q^\*(a)$. We estimate $Q_t(a)\approx q^\*(a)$ by the **incremental sample mean**
$$Q_{n}(a)=Q_{n-1}(a)+\tfrac{1}{n}\big(R_n-Q_{n-1}(a)\big),$$
which is exactly the running average in $O(1)$ memory (derivable from $\bar X_n=\tfrac1n\sum_{i\le n}R_i$).

We compare three principled strategies:

### II.1 $\epsilon$-greedy
$$a_t=\begin{cases}\text{uniform random arm} & \text{w.p. }\epsilon\\\arg\max_a Q_t(a) & \text{w.p. }1-\epsilon\end{cases}$$
Simple, but explores **uniformly** even arms it already knows are bad.

### II.2 UCB1 (optimism in the face of uncertainty)
Pick the arm with the highest **upper confidence bound**:
$$a_t=\arg\max_a\Big[Q_t(a)+c\sqrt{\tfrac{\ln t}{N_t(a)}}\Big].$$
The bonus is a Hoeffding confidence radius: it shrinks as $N_t(a)$ grows, so exploration is *directed* toward under-sampled arms.

### II.3 Thompson Sampling (Bayesian / probability matching)
Maintain a posterior over each arm's mean and **sample** from it. For Bernoulli rewards the conjugate prior is $\text{Beta}(\alpha,\beta)$:
$$\theta_a\sim\text{Beta}(\alpha_a,\beta_a),\quad a_t=\arg\max_a\theta_a,$$
then update $\alpha_a\!\mathrel{+}=\!r,\ \beta_a\!\mathrel{+}=\!(1-r)$. We explore in proportion to the probability an arm is optimal.

In [ ]:
# ── Three exploration strategies on the SAME Gaussian bandit, compared by regret ──
class GaussianBandit:
    # k arms, reward ~ N(q*(a), 1). Tracks cumulative regret.
    def __init__(self, k=10, seed=0):
        rng = np.random.default_rng(seed)
        self.k = k
        self.q_star = rng.normal(0, 1, k)
        self.best = self.q_star.max()
    def pull(self, a, rng):
        return rng.normal(self.q_star[a], 1.0)

def run_epsilon_greedy(bandit, T, eps, rng):
    Q, N, regret = np.zeros(bandit.k), np.zeros(bandit.k), []
    cum = 0.0
    for _ in range(T):
        a = rng.integers(bandit.k) if rng.random() < eps else int(np.argmax(Q))
        r = bandit.pull(a, rng); N[a] += 1; Q[a] += (r - Q[a]) / N[a]
        cum += bandit.best - bandit.q_star[a]; regret.append(cum)
    return np.array(regret)

def run_ucb1(bandit, T, c, rng):
    Q, N, regret = np.zeros(bandit.k), np.zeros(bandit.k), []
    cum = 0.0
    for t in range(1, T + 1):
        if (N == 0).any():
            a = int(np.argmin(N))                      # pull each arm once first
        else:
            a = int(np.argmax(Q + c * np.sqrt(np.log(t) / N)))
        r = bandit.pull(a, rng); N[a] += 1; Q[a] += (r - Q[a]) / N[a]
        cum += bandit.best - bandit.q_star[a]; regret.append(cum)
    return np.array(regret)

def run_thompson_gaussian(bandit, T, rng):
    # Gaussian posterior on each mean (known noise var=1): mu ~ N(mean_a, 1/N_a)
    sum_r, N, regret = np.zeros(bandit.k), np.zeros(bandit.k), []
    cum = 0.0
    for _ in range(T):
        means = np.where(N > 0, sum_r / np.maximum(N, 1), 0.0)
        samples = rng.normal(means, 1.0 / np.sqrt(np.maximum(N, 1)))
        a = int(np.argmax(samples))
        r = bandit.pull(a, rng); N[a] += 1; sum_r[a] += r
        cum += bandit.best - bandit.q_star[a]; regret.append(cum)
    return np.array(regret)

# Average regret over many independent problems for a fair comparison
T, n_runs = 1000, 200
agg = {"eps-greedy (0.1)": [], "UCB1 (c=2)": [], "Thompson": []}
for run in range(n_runs):
    bandit = GaussianBandit(k=10, seed=run)
    rng = np.random.default_rng(1000 + run)
    agg["eps-greedy (0.1)"].append(run_epsilon_greedy(bandit, T, 0.1, rng))
    agg["UCB1 (c=2)"].append(run_ucb1(bandit, T, 2.0, rng))
    agg["Thompson"].append(run_thompson_gaussian(bandit, T, rng))

plt.figure(figsize=(9, 4.5))
for label, runs in agg.items():
    plt.plot(np.mean(runs, axis=0), lw=2, label=label)
plt.xlabel("step $t$"); plt.ylabel("cumulative regret")
plt.title(f"Average regret over {n_runs} random 10-armed bandits (lower is better)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for label, runs in agg.items():
    print(f"{label:20s} final regret = {np.mean(runs, axis=0)[-1]:6.2f}")

### II.4 Watching the Bayesian posterior concentrate

To make the *probability distribution* in Thompson Sampling concrete, we run a **Bernoulli** bandit (each arm pays 0/1 with unknown probability $p_a$) and plot the $\text{Beta}(\alpha_a,\beta_a)$ posterior for every arm. Initially each posterior is the flat $\text{Beta}(1,1)=\text{Uniform}[0,1]$ prior; as evidence accumulates the densities sharpen around the true $p_a$ (red lines). The arm that is *probably* best gets sampled most, so its posterior is the narrowest.

In [ ]:
# ── Beta-Bernoulli Thompson Sampling: posterior evolution ──
true_p = np.array([0.25, 0.40, 0.55, 0.75, 0.50])   # unknown to the agent
k = len(true_p)
alpha, betap = np.ones(k), np.ones(k)               # Beta(1,1) priors
rng = np.random.default_rng(SEED)

snapshots = {}
T = 600
for t in range(1, T + 1):
    theta = rng.beta(alpha, betap)        # sample a plausible mean per arm
    a = int(np.argmax(theta))             # act greedily on the sample
    r = 1 if rng.random() < true_p[a] else 0
    alpha[a] += r; betap[a] += 1 - r      # conjugate Beta update
    if t in (5, 50, 600):
        snapshots[t] = (alpha.copy(), betap.copy())

xs = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, len(snapshots), figsize=(15, 4), sharey=True)
for ax, (t, (al, be)) in zip(axes, snapshots.items()):
    for i in range(k):
        ax.plot(xs, beta_dist.pdf(xs, al[i], be[i]), lw=2, label=f"arm {i}")
        ax.axvline(true_p[i], color="red", ls=":", alpha=0.4)
    ax.set_title(f"after t = {t} pulls"); ax.set_xlabel("$\\theta$ (arm mean)")
axes[0].set_ylabel("posterior density"); axes[0].legend(fontsize=8)
plt.suptitle("Thompson Sampling: Beta posteriors concentrate on the true means (red)", y=1.03)
plt.tight_layout(); plt.show()

best = int(np.argmax(true_p))
pulls = (alpha + betap - 2)
print(f"True best arm = {best} (p={true_p[best]}).  Pulls per arm: {pulls.astype(int)}")
print(f"Fraction of pulls on the best arm: {pulls[best]/pulls.sum():.1%}")

---
# Part III — Exact Solution Methods (Dynamic Programming)

*Lecture 1 — Pieter Abbeel.* When the MDP $(\mathcal S,\mathcal A,\mathcal P,\mathcal R,\gamma)$ is **fully known and small**, we can solve for $V^\*$ exactly by turning the Bellman optimality equation into an iterative update. Two classic algorithms:

### III.1 Value Iteration
Repeatedly apply the **Bellman optimality operator** $\mathcal T^\*$:
$$V_{k+1}(s)=\underbrace{\max_a\sum_{s'}\mathcal P(s'\mid s,a)\big[\mathcal R(s,a,s')+\gamma V_k(s')\big]}_{(\mathcal T^\* V_k)(s)}.$$
$\mathcal T^\*$ is a **$\gamma$-contraction** in the $\|\cdot\|_\infty$ norm, so by the Banach fixed-point theorem $V_k\to V^\*$ geometrically: $\|V_k-V^\*\|_\infty\le\gamma^k\|V_0-V^\*\|_\infty$.

### III.2 Policy Iteration
Alternate two steps until the policy stops changing:
1. **Policy evaluation** — solve the linear system $V^\pi=\mathcal R^\pi+\gamma\mathcal P^\pi V^\pi$ (here by iteration).
2. **Policy improvement** — act greedily: $\pi'(s)=\arg\max_a\sum_{s'}\mathcal P(s'\mid s,a)[\mathcal R+\gamma V^\pi(s')]$.
Guaranteed to reach $\pi^\*$ in a **finite** number of iterations for finite MDPs.

**Environment.** We use Gymnasium's `FrozenLake-v1` (4×4, deterministic) and read its built-in transition model `env.P[s][a] = [(prob, s', reward, done), …]` — i.e. the exact kernel $\mathcal P$ — so both algorithms are truly model-based.

In [ ]:
# ── Value Iteration and Policy Iteration on FrozenLake (exact, model-based) ──
env = gym.make("FrozenLake-v1", is_slippery=False)
P = env.unwrapped.P            # P[s][a] = list of (prob, s_next, reward, done)
nS, nA = env.observation_space.n, env.action_space.n
GAMMA = 0.99

def bellman_backup(s, V):
    # Return the vector of Q(s,a) under current V using the true model P.
    q = np.zeros(nA)
    for a in range(nA):
        for prob, s2, r, done in P[s][a]:
            q[a] += prob * (r + GAMMA * V[s2] * (not done))
    return q

def value_iteration(theta=1e-9):
    V = np.zeros(nS); deltas = []
    while True:
        delta = 0.0
        for s in range(nS):
            v_old = V[s]
            V[s] = bellman_backup(s, V).max()      # T* operator
            delta = max(delta, abs(v_old - V[s]))
        deltas.append(delta)
        if delta < theta:
            break
    policy = np.array([bellman_backup(s, V).argmax() for s in range(nS)])
    return V, policy, deltas

def policy_iteration():
    policy = np.zeros(nS, dtype=int); iters = 0
    while True:
        # 1. Policy evaluation (iterative)
        V = np.zeros(nS)
        while True:
            delta = 0.0
            for s in range(nS):
                v_old = V[s]
                V[s] = bellman_backup(s, V)[policy[s]]
                delta = max(delta, abs(v_old - V[s]))
            if delta < 1e-9:
                break
        # 2. Policy improvement
        new_policy = np.array([bellman_backup(s, V).argmax() for s in range(nS)])
        iters += 1
        if np.array_equal(new_policy, policy):
            return V, policy, iters
        policy = new_policy

V_vi, pi_vi, deltas = value_iteration()
V_pi, pi_pi, n_iters = policy_iteration()

print(f"Value Iteration converged in {len(deltas)} sweeps.")
print(f"Policy Iteration converged in {n_iters} policy updates.")
print(f"Both agree on the optimal policy: {np.array_equal(pi_vi, pi_pi)}")

# Visualise V* and the greedy policy
arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
desc = env.unwrapped.desc.astype(str).ravel()      # 'S','F','H','G'
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
im = ax[0].imshow(V_vi.reshape(4, 4), cmap="viridis")
ax[0].set_title("$V^*(s)$ from Value Iteration")
for s in range(nS):
    ax[0].text(s % 4, s // 4, f"{V_vi[s]:.2f}", ha="center", va="center", color="white")
plt.colorbar(im, ax=ax[0], fraction=0.046)

ax[1].imshow(np.zeros((4, 4)), cmap="Greys", vmin=0, vmax=1)
ax[1].set_title("Greedy optimal policy $\\pi^*$")
for s in range(nS):
    label = "G" if desc[s] == "G" else "H" if desc[s] == "H" else arrows[pi_vi[s]]
    color = "green" if desc[s] == "G" else "red" if desc[s] == "H" else "black"
    ax[1].text(s % 4, s // 4, label, ha="center", va="center", fontsize=18, color=color)
ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout(); plt.show()

# Convergence is geometric, as the contraction theory predicts
plt.figure(figsize=(7, 3))
plt.semilogy(deltas, "o-"); plt.xlabel("sweep"); plt.ylabel("$\\|V_{k+1}-V_k\\|_\\infty$")
plt.title("Value Iteration: geometric ($\\gamma$-contraction) convergence")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
# Part IV — Model-Free Learning

*Lecture 2 — Rocky Duan.* Now we **drop the assumption that $\mathcal P$ and $\mathcal R$ are known.** The agent must learn purely from sampled transitions $(s,a,r,s')$. Two ideas, then two control algorithms.

### IV.1 Monte Carlo (MC) vs Temporal Difference (TD)
Both estimate $V^\pi$ from experience, differing in the *target* they regress toward:

- **MC** waits for the full return: $\;V(s)\leftarrow V(s)+\alpha\big(G_t-V(s)\big).$ Unbiased, but high variance and needs episodes to terminate.
- **TD(0)** bootstraps off its own next estimate: $\;V(s)\leftarrow V(s)+\alpha\big(\underbrace{r+\gamma V(s')}_{\text{TD target}}-V(s)\big).$ The bracket is the **TD error** $\delta_t$. Biased (uses an estimate) but low variance and works online, per step.

### IV.2 Control: SARSA (on-policy) vs Q-Learning (off-policy)
Both learn $Q(s,a)$ with an $\epsilon$-greedy behaviour policy, but use different targets:

$$\textbf{SARSA: }\;Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma Q(s',a')-Q(s,a)\big]$$
where $a'$ is the action **actually taken** next → evaluates the policy it follows (on-policy).

$$\textbf{Q-Learning: }\;Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big]$$
uses the **greedy** next action regardless of behaviour → learns $Q^\*$ directly (off-policy).

We train both on the **slippery** FrozenLake (stochastic $\mathcal P$) so the on-/off-policy distinction matters.

In [ ]:
# ── SARSA vs Q-Learning on slippery FrozenLake (model-free, tabular) ──
def make_env():
    return gym.make("FrozenLake-v1", is_slippery=True)

def epsilon_greedy(Q, s, eps, nA, rng):
    return rng.integers(nA) if rng.random() < eps else int(np.argmax(Q[s]))

def train_td_control(method, episodes=20000, alpha=0.1, gamma=0.99,
                     eps_start=1.0, eps_end=0.05):
    env = make_env()
    nS, nA = env.observation_space.n, env.action_space.n
    Q = np.zeros((nS, nA))
    rng = np.random.default_rng(SEED)
    returns = []
    for ep in range(episodes):
        eps = max(eps_end, eps_start * (1 - ep / episodes))   # linear decay
        s, _ = env.reset(seed=int(rng.integers(1e9)))
        a = epsilon_greedy(Q, s, eps, nA, rng)
        done, G = False, 0.0
        while not done:
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            a2 = epsilon_greedy(Q, s2, eps, nA, rng)
            if method == "sarsa":
                target = r + gamma * Q[s2, a2] * (not term)
            else:  # q-learning
                target = r + gamma * np.max(Q[s2]) * (not term)
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
            G += r
        returns.append(G)
    return Q, np.array(returns)

Q_sarsa, ret_sarsa = train_td_control("sarsa")
Q_ql, ret_ql = train_td_control("qlearning")

def smooth(x, w=500):
    return pd.Series(x).rolling(w, min_periods=1).mean().values

plt.figure(figsize=(9, 4))
plt.plot(smooth(ret_sarsa), label="SARSA (on-policy)", lw=2)
plt.plot(smooth(ret_ql), label="Q-Learning (off-policy)", lw=2)
plt.xlabel("episode"); plt.ylabel("success rate (500-ep moving avg)")
plt.title("SARSA vs Q-Learning on slippery FrozenLake")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"SARSA      final success rate: {smooth(ret_sarsa)[-1]:.2%}")
print(f"Q-Learning final success rate: {smooth(ret_ql)[-1]:.2%}")
print("On slippery ice, SARSA's on-policy target makes it more conservative/safe.")

---
# Part V — Deep Reinforcement Learning

When $\mathcal S$ is large or continuous, we replace tables with **function approximators** $f_\theta$ (neural nets). We cover the two main families.

### V.1 Value-based: DQN
*Lecture 3 — Vlad Mnih.* Approximate $Q^\*(s,a)\approx Q_\theta(s,a)$ and minimise the **mean-squared Bellman error** on sampled transitions:
$$\mathcal L(\theta)=\mathbb E_{(s,a,r,s')\sim\mathcal D}\Big[\big(r+\gamma\max_{a'}Q_{\theta^-}(s',a')-Q_\theta(s,a)\big)^2\Big].$$
Two stabilisers make this work: (i) an **experience replay** buffer $\mathcal D$ that decorrelates samples, and (ii) a slowly-updated **target network** $\theta^-$ so the regression target doesn't chase itself. **DQN requires a discrete action space** (it maxes over actions).

### V.2 Policy-based: REINFORCE → A2C → PPO
*Lectures 4–5 — Abbeel & Schulman.* Directly optimise $J(\theta)=\mathbb E_{\tau\sim\pi_\theta}[G_0]$ via the **policy-gradient theorem**:
$$\nabla_\theta J(\theta)=\mathbb E_{\pi_\theta}\big[\nabla_\theta\log\pi_\theta(a_t\mid s_t)\,\Psi_t\big],$$
where the weight $\Psi_t$ trades bias/variance:
- **REINFORCE**: $\Psi_t=G_t$ (Monte-Carlo return; unbiased, high variance).
- **A2C**: $\Psi_t=A(s_t,a_t)=Q(s_t,a_t)-V(s_t)$ (advantage; a learned critic $V_\phi$ slashes variance).
- **PPO**: maximise a **clipped surrogate** that keeps the new policy close to the old one. With ratio $r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$,
$$\mathcal L^{\text{CLIP}}(\theta)=\mathbb E_t\Big[\min\big(r_t A_t,\ \mathrm{clip}(r_t,1-\epsilon,1+\epsilon)A_t\big)\Big].$$
PPO supports **both** discrete and continuous actions and is the practical default.

We first implement **REINFORCE from scratch** (so the gradient is fully transparent), then train **DQN/A2C/PPO via Stable-Baselines3** for a controlled comparison on `CartPole-v1`.

In [ ]:
# ── REINFORCE from scratch (PyTorch) on CartPole-v1 ──
# Makes the policy-gradient estimator fully explicit: loss = -E[ log pi(a|s) * G_t ].
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, n_actions)          # logits → Categorical
        )
    def forward(self, x):
        return self.net(x)

def train_reinforce(episodes=600, gamma=0.99, lr=2e-3):
    env = gym.make("CartPole-v1")
    torch.manual_seed(SEED)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    ep_returns = []
    for ep in range(episodes):
        s, _ = env.reset(seed=SEED + ep)
        log_probs, rewards = [], []
        done = False
        while not done:
            logits = policy(torch.as_tensor(s, dtype=torch.float32))
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            log_probs.append(dist.log_prob(a))
            s, r, term, trunc, _ = env.step(a.item())
            rewards.append(r); done = term or trunc
        # Discounted returns, then standardised as a variance-reduction baseline
        G, returns = 0.0, []
        for r in reversed(rewards):
            G = r + gamma * G; returns.append(G)
        returns = torch.tensor(returns[::-1], dtype=torch.float32)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        loss = -(torch.stack(log_probs) * returns).sum()   # policy-gradient loss
        opt.zero_grad(); loss.backward(); opt.step()
        ep_returns.append(sum(rewards))
    return np.array(ep_returns)

reinforce_returns = train_reinforce()
plt.figure(figsize=(9, 3.5))
plt.plot(reinforce_returns, alpha=0.3, color="#4C72B0")
plt.plot(smooth(reinforce_returns, 20), color="#4C72B0", lw=2, label="20-ep avg")
plt.axhline(500, color="red", ls="--", alpha=0.6, label="max return")
plt.xlabel("episode"); plt.ylabel("episode return"); plt.legend()
plt.title("REINFORCE (from scratch) on CartPole-v1"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Mean return over last 50 episodes: {reinforce_returns[-50:].mean():.1f}")

### V.3 Controlled comparison: DQN vs A2C vs PPO (Stable-Baselines3)

We now train the three reference algorithms on the **same** environment, the same budget, and the same seed, then evaluate each with `evaluate_policy`. Why Stable-Baselines3 rather than hand-rolling each one? Because the goal here is a **fair, bug-free comparison**: SB3's implementations are extensively unit-tested and benchmark-matched, so any performance difference reflects the *algorithm*, not an implementation bug on our part.

Expected qualitative outcome on `CartPole-v1`:
- **DQN** — sample-efficient (replay reuses data) but can be jumpy.
- **A2C** — fast wall-clock per step, on-policy, moderate variance.
- **PPO** — the clipped objective gives the most stable curve and usually saturates at the 500 cap.

In [ ]:
# ── Train DQN, A2C, PPO on CartPole-v1 under an identical budget, then evaluate ──
TIMESTEPS = 30_000

def train_sb3(AlgoCls, **kwargs):
    env = gym.make("CartPole-v1")
    model = AlgoCls("MlpPolicy", env, seed=SEED, verbose=0, device="cpu", **kwargs)
    model.learn(total_timesteps=TIMESTEPS)
    mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20)
    return model, mean_r, std_r

configs = {
    "DQN": (DQN, dict(learning_rate=1e-3, buffer_size=50_000, learning_starts=1000,
                       target_update_interval=500, exploration_fraction=0.2)),
    "A2C": (A2C, dict(learning_rate=7e-4, n_steps=5)),
    "PPO": (PPO, dict(learning_rate=3e-4, n_steps=1024, batch_size=64,
                       n_epochs=10, clip_range=0.2)),
}

results = {}
trained = {}
for name, (cls, kw) in configs.items():
    print(f"Training {name} for {TIMESTEPS:,} steps …")
    model, mean_r, std_r = train_sb3(cls, **kw)
    results[name] = (mean_r, std_r); trained[name] = model
    print(f"   {name}: mean eval return = {mean_r:.1f} ± {std_r:.1f}")

# Bar chart of evaluation performance
names = list(results)
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]
plt.figure(figsize=(7, 4))
bars = plt.bar(names, means, yerr=stds, capsize=8,
               color=["#55A868", "#DD8452", "#4C72B0"], edgecolor="black")
plt.axhline(500, color="red", ls="--", alpha=0.6, label="max return (500)")
for b, m in zip(bars, means):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 8, f"{m:.0f}",
             ha="center", fontweight="bold")
plt.ylabel("mean evaluation return (20 episodes)")
plt.title(f"DQN vs A2C vs PPO on CartPole-v1 ({TIMESTEPS:,} steps)")
plt.legend(); plt.ylim(0, 560); plt.tight_layout(); plt.show()

---
# Part VI — Animating the Agent's Internal State

A learning curve tells you *that* an agent improved; it does not show you *how it decides*. Here we roll out the **trained PPO agent** on `CartPole-v1` and animate two synchronised views, frame by frame:

1. **The physical state** $s_t=(x,\dot x,\theta,\dot\theta)$ — cart position and pole angle, drawn directly from the observation vector.
2. **The agent's internal decision** — the policy distribution $\pi_\theta(\cdot\mid s_t)$ over $\{\text{left},\text{right}\}$, extracted from PPO's network, plus the value estimate $V_\phi(s_t)$.

This makes the abstract objects from Part I — the **state vector** and the **action probability distribution** — visible and time-resolved. We extract the probabilities from the SB3 policy by calling its underlying PyTorch distribution, so the animation reflects the *actual* internal state, not a re-implementation.

In [ ]:
# ── Roll out the trained PPO agent and record state + internal policy distribution ──
ppo_model = trained["PPO"]
roll_env = gym.make("CartPole-v1")

def policy_distribution(model, obs):
    # Return action probabilities and value estimate from SB3's PyTorch policy.
    obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        dist = model.policy.get_distribution(obs_t)
        probs = dist.distribution.probs.squeeze(0).numpy()
        value = model.policy.predict_values(obs_t).item()
    return probs, value

# Record a full episode
states, probs_hist, values, actions = [], [], [], []
obs, _ = roll_env.reset(seed=SEED)
for _ in range(300):
    probs, value = policy_distribution(ppo_model, obs)
    a, _ = ppo_model.predict(obs, deterministic=True)
    states.append(obs.copy()); probs_hist.append(probs)
    values.append(value); actions.append(int(a))
    obs, r, term, trunc, _ = roll_env.step(int(a))
    if term or trunc:
        break
states = np.array(states); probs_hist = np.array(probs_hist); values = np.array(values)
print(f"Recorded {len(states)} steps. Pole survived {len(states)} / 300 frames.")

In [ ]:
# ── Build the synchronised animation (physical state + internal policy state) ──
from matplotlib.animation import FuncAnimation

fig = plt.figure(figsize=(12, 4.5))
gs = fig.add_gridspec(1, 3, width_ratios=[2, 1, 1])
ax_cart = fig.add_subplot(gs[0])   # physical cart-pole
ax_pi   = fig.add_subplot(gs[1])   # policy distribution pi(a|s)
ax_val  = fig.add_subplot(gs[2])   # value estimate over time

CART_W, CART_H, POLE_LEN = 0.4, 0.25, 1.0

def draw_static_cart(ax):
    ax.set_xlim(-2.6, 2.6); ax.set_ylim(-0.6, 1.6)
    ax.set_aspect("equal"); ax.set_yticks([])
    ax.axhline(0, color="black", lw=1)
    ax.set_title("Physical state $s_t=(x,\\dot x,\\theta,\\dot\\theta)$")

cart_patch = Rectangle((0, 0), CART_W, CART_H, fc="#4C72B0", ec="black")
pole_line, = ax_cart.plot([], [], lw=5, color="#C44E52", solid_capstyle="round")
info_text = ax_cart.text(-2.5, 1.4, "", fontsize=9, va="top")

pi_bars = ax_pi.bar(["left", "right"], [0.5, 0.5], color=["#55A868", "#DD8452"], ec="black")
ax_pi.set_ylim(0, 1); ax_pi.set_title("Internal policy $\\pi_\\theta(a|s_t)$")
ax_pi.set_ylabel("probability")

val_line, = ax_val.plot([], [], color="#8172B3", lw=2)
ax_val.set_xlim(0, len(states)); ax_val.set_ylim(values.min() - 1, values.max() + 1)
ax_val.set_title("Critic value $V_\\phi(s_t)$"); ax_val.set_xlabel("step")
ax_val.grid(alpha=0.3)

def init():
    draw_static_cart(ax_cart)
    ax_cart.add_patch(cart_patch)
    return cart_patch, pole_line, info_text, *pi_bars, val_line

def update(i):
    x, x_dot, theta, theta_dot = states[i]
    # Cart
    cart_patch.set_xy((x - CART_W / 2, 0))
    # Pole (theta measured from vertical)
    px = [x, x + POLE_LEN * np.sin(theta)]
    py = [CART_H, CART_H + POLE_LEN * np.cos(theta)]
    pole_line.set_data(px, py)
    info_text.set_text(f"step {i}\nx={x:+.2f}  θ={np.degrees(theta):+.1f}°\n"
                       f"action={'right' if actions[i] else 'left'}")
    # Policy distribution
    for bar, p in zip(pi_bars, probs_hist[i]):
        bar.set_height(p)
    # Value trace
    val_line.set_data(np.arange(i + 1), values[:i + 1])
    return cart_patch, pole_line, info_text, *pi_bars, val_line

anim = FuncAnimation(fig, update, frames=len(states), init_func=init,
                     blit=True, interval=40)
plt.close(fig)   # prevent duplicate static figure; HTML below shows the player
HTML(anim.to_jshtml())

### VI.2 Internal state of a *tabular* agent

The PPO animation showed a neural agent in a continuous state space. For contrast, the cell below animates the **Q-Learning agent from Part IV** walking the FrozenLake grid. Here the "internal state" is the **Q-table row** $Q(s,\cdot)$ for the current cell — we highlight the agent's position and draw the greedy action it derives from its learned values. This connects the discrete-MDP theory (Parts I–IV) to a moving picture of a policy in action.

In [ ]:
# ── Animate the trained Q-Learning agent traversing FrozenLake ──
# Use the deterministic lake so the greedy policy gives a clean path to visualise.
viz_env = gym.make("FrozenLake-v1", is_slippery=False)
# Re-derive a greedy policy on the deterministic lake for a tidy trajectory
Q_det, _ = train_td_control("qlearning", episodes=8000)
greedy = np.argmax(Q_det, axis=1)
desc = viz_env.unwrapped.desc.astype(str).ravel()

# Record a greedy rollout
path = []
s, _ = viz_env.reset(seed=SEED)
for _ in range(50):
    path.append(s)
    s, r, term, trunc, _ = viz_env.step(int(greedy[s]))
    if term or trunc:
        path.append(s); break

arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
fig, (axg, axq) = plt.subplots(1, 2, figsize=(11, 4.6))

def draw_grid():
    axg.clear()
    axg.set_xlim(-0.5, 3.5); axg.set_ylim(-0.5, 3.5); axg.invert_yaxis()
    axg.set_xticks([]); axg.set_yticks([]); axg.set_title("Q-Learning agent on FrozenLake")
    for cell in range(16):
        r, c = cell // 4, cell % 4
        ch = desc[cell]
        face = {"S": "#C7E9C0", "F": "white", "H": "#4C72B0", "G": "#55A868"}[ch]
        axg.add_patch(Rectangle((c - 0.5, r - 0.5), 1, 1, fc=face, ec="gray"))
        if ch not in ("H", "G"):
            axg.text(c, r, arrows[greedy[cell]], ha="center", va="center",
                     fontsize=14, color="black", alpha=0.4)
        if ch in ("H", "G"):
            axg.text(c, r, ch, ha="center", va="center", fontsize=14,
                     color="white", fontweight="bold")

agent_dot = Circle((0, 0), 0.22, fc="#DD8452", ec="black", zorder=5)

def update_grid(i):
    draw_grid(); axg.add_patch(agent_dot)
    s = path[i]; r, c = s // 4, s % 4
    agent_dot.center = (c, r)
    # Internal state: the Q-row for the current cell
    axq.clear()
    axq.bar([arrows[a] for a in range(4)], Q_det[s], color="#4C72B0", ec="black")
    axq.set_ylim(0, max(Q_det.max(), 1e-3))
    axq.set_title(f"Internal state: $Q(s={s},\\cdot)$")
    axq.set_ylabel("action value")
    return ()

anim2 = FuncAnimation(fig, update_grid, frames=len(path), interval=600)
plt.close(fig)
HTML(anim2.to_jshtml())

---
# Summary & Where to Go Next

**What we built, top to bottom:**

| Part | Entity made precise | Experiment |
|------|--------------------|-----------|
| I | Probability space, action/state spaces, kernel $\mathcal P$, $V/Q$, Bellman | Distribution & return plots |
| II | Exploration as probability matching | $\epsilon$-greedy vs UCB1 vs Thompson (regret) + Beta posteriors |
| III | Bellman optimality operator as a $\gamma$-contraction | Value & Policy Iteration on FrozenLake |
| IV | TD error, on- vs off-policy targets | SARSA vs Q-Learning on slippery ice |
| V | Policy-gradient theorem, clipped surrogate | REINFORCE from scratch; DQN/A2C/PPO via SB3 |
| VI | State vector & policy distribution over time | PPO cart-pole + Q-Learning grid animations |

**Key takeaways**
- The **policy is a conditional distribution** $\pi(a\mid s)$; everything else (value functions, gradients) is an expectation over the trajectory measure $\mathbb P_\pi$.
- Dynamic programming needs the model; **TD/Q-Learning** learn from samples; **deep RL** adds function approximation when tables no longer fit.
- **PPO** is the robust default: discrete *and* continuous actions, stable via clipping.

**Recommended further reading (from the Readme references):**
- Berkeley Deep RL Bootcamp lecture slides (MDPs → TRPO/PPO).
- OpenAI **Spinning Up** — code + math for policy-gradient methods.
- Sutton & Barto, *Reinforcement Learning: An Introduction* — the canonical theory text.
- **CleanRL** / **Stable-Baselines3** docs — single-file and production-grade implementations.

*This notebook is runnable end-to-end. Run Cell 1 (pinned install) once, restart the kernel, then run all.*